In [32]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
import joblib

In [33]:
DATASET_PATH = "../dataset/flows.csv"
PARQUET_DATASET_PATH = "../dataset/bccc-cpacket-cloud-ddos-2024-merged.parquet"

pq = pd.read_parquet(PARQUET_DATASET_PATH)
df = pd.read_csv(DATASET_PATH)

df.head()

,id,expiration_id,src_ip,src_mac,src_oui,src_port,dst_ip,dst_mac,dst_oui,dst_port,...,down_up_rate,bwd_avg_segment_size,handshake_state,delta_start,mode_payload_bytes_delta_len,skewness_payload_bytes_delta_len,skewness_fwd_payload_bytes_delta_len,skewness_bwd_payload_bytes_delta_len,duration,packets_count
0,0,0,127.0.0.1,00:00:00:00:00:00,00:00:00,34466,127.0.0.1,00:00:00:00:00:00,00:00:00,37755,...,3.570552,194.0,0.0,0.0,0.0,1.375929,0.0,0.607926,1,5
1,1,0,127.0.0.1,00:00:00:00:00:00,00:00:00,53518,127.0.0.1,00:00:00:00:00:00,00:00:00,37755,...,2.000000,66.0,0.0,0.0,0.0,0.000000,0.0,0.000000,2049,3
2,2,0,127.0.0.1,00:00:00:00:00:00,00:00:00,51498,127.0.0.1,00:00:00:00:00:00,00:00:00,37535,...,66.000000,66.0,0.0,0.0,0.0,0.000000,0.0,0.000000,1,1
3,3,0,127.0.0.1,00:00:00:00:00:00,00:00:00,37755,127.0.0.1,00:00:00:00:00:00,00:00:00,34466,...,66.000000,66.0,0.0,0.0,0.0,0.000000,0.0,0.000000,1,1
4,4,0,127.0.0.1,00:00:00:00:00:00,00:00:00,37755,127.0.0.1,00:00:00:00:00:00,00:00:00,53528,...,1.500000,66.0,0.0,0.0,0.0,0.000000,0.0,0.000000,15360,5


In [34]:
print("Parquet Dataset:")
print(pq.head())

Parquet Dataset:
   src_port  dst_port  duration  packets_count  fwd_packets_count  \
0     54573     25094  0.000063              3                  2   
1     25094     54573  0.000000              1                  0   
2     54573     25094  0.000028              3                  1   
3      9147     18060  0.000055              3                  2   
4     18060      9147  0.000000              1                  0   

   bwd_packets_count  total_payload_bytes  fwd_total_payload_bytes  \
0                  1                    0                        0   
1                  1                    0                        0   
2                  2                    0                        0   
3                  1                    0                        0   
4                  1                    0                        0   

   bwd_total_payload_bytes  payload_bytes_max  ...  \
0                        0                  0  ...   
1                        0             

In [35]:
df.shape

(82, 445)

In [41]:
import joblib
cols = df.columns
set_cols = set(cols)

features_columns = joblib.load("./logisticRegression_preprocess_info.joblib")["feature_columns"]
print(f"len of features columns: {len(features_columns)}")

set_feature = set(features_columns)
difference = set_cols - set_feature
same_cols = set_cols & set_feature
print(f"Columns in both dataset and features: {len(same_cols)}")
print(f"Columns in dataset but not in features: {len(difference)}")
cols_to_drop = list(set_cols - set_feature)
print(f"Total columns in features: {len(features_columns)}")
print(f"Total columns in dataset: {len(cols)}")
print(f"Columns to drop: {len(cols_to_drop)}")


len of features columns: 162
Columns in both dataset and features: 162
Columns in dataset but not in features: 283
Total columns in features: 162
Total columns in dataset: 445
Columns to drop: 283


In [37]:
print("Is 'bwd' in set_cols?", "bwd" in set_cols)

Is 'bwd' in set_cols? False


In [38]:
for col in set_cols:
    print(col)

fwd_packets_IAT_cov
splt_direction
packets_delta_len_total
min_header_bytes
avg_fwd_bytes_per_bulk
dst2src_fin_packets
fwd_payload_bytes_skewness
std_fwd_header_bytes_delta_len
dst2src_ack_packets
packets_IAT_kurtosis
cwr_flag_percentage_in_total
fwd_total_payload_bytes
cov_packets_delta_time
src2dst_stddev_piat_ms
median_header_bytes_delta_len
fwd_fin_flag_percentage_in_fwd_packets
idle_count
packets_IAT_cov
bwd_mode_header_bytes
bidirectional_cwr_packets
bidirectional_psh_packets
bwd_bulk_count
protocol
fwd_psh_flag_counts
bwd_payload_bytes_mean
bwd_packets_delta_len_count
bwd_rst_cnt
bwd_payload_bytes_cov
dst2src_first_seen_ms
kurtosis_bwd_header_bytes_delta_len
bytes_rate
fwd_syn_cnt
bwd_bulk_total_dur_ms
payload_bytes_std
bwd_syn_flag_percentage_in_total
fwd_win_std
src_ip
packets_IAT_max
packets_IAT_mean
bwd_packets_IAT_median
skewness_header_bytes_delta_len
fwd_win_mean
fwd_ack_cnt
kurtosis_fwd_packets_delta_time
bwd_payload_bytes_max
mean_bwd_packets_delta_len
bwd_min_header_by

In [39]:
for col in same_cols:
    print(col)

fwd_packets_IAT_cov
urg_flag_counts
min_bwd_packets_delta_len
bwd_packets_rate
avg_fwd_bytes_per_bulk
variance_packets_delta_len
fwd_packets_IAT_skewness
min_fwd_packets_delta_time
fwd_payload_bytes_skewness
bwd_syn_flag_counts
bwd_max_header_bytes
cwr_flag_percentage_in_total
fwd_total_payload_bytes
skewness_bwd_packets_delta_len
min_fwd_packets_delta_len
median_fwd_packets_delta_len
cov_fwd_packets_delta_len
skewness_fwd_packets_delta_len
subflow_fwd_bytes
ece_flag_counts
bwd_rst_flag_percentage_in_total
variance_fwd_header_bytes_delta_len
handshake_state
fwd_payload_bytes_cov
skewness_fwd_payload_bytes_delta_len
syn_flag_percentage_in_total
bwd_bytes_rate
subflow_fwd_packets
mode_packets_delta_len
bwd_packets_IAT_skewness
src_port
psh_flag_percentage_in_total
packets_IAT_cov
variance_fwd_packets_delta_len
variance_bwd_header_bytes_delta_len
max_bwd_header_bytes_delta_len
rst_flag_percentage_in_total
max_fwd_packets_delta_len
max_header_bytes_delta_len
bwd_ece_flag_percentage_in_tota

In [40]:
for col in set_feature:
    print(col)

fwd_packets_IAT_cov
urg_flag_counts
min_bwd_packets_delta_len
bwd_packets_rate
avg_fwd_bytes_per_bulk
variance_packets_delta_len
fwd_packets_IAT_skewness
min_fwd_packets_delta_time
fwd_payload_bytes_skewness
bwd_syn_flag_counts
bwd_max_header_bytes
cwr_flag_percentage_in_total
fwd_total_payload_bytes
skewness_bwd_packets_delta_len
min_fwd_packets_delta_len
median_fwd_packets_delta_len
skewness_fwd_packets_delta_len
cov_fwd_packets_delta_len
subflow_fwd_bytes
ece_flag_counts
bwd_rst_flag_percentage_in_total
variance_fwd_header_bytes_delta_len
handshake_state
fwd_payload_bytes_cov
skewness_fwd_payload_bytes_delta_len
syn_flag_percentage_in_total
bwd_bytes_rate
subflow_fwd_packets
mode_packets_delta_len
bwd_packets_IAT_skewness
src_port
psh_flag_percentage_in_total
packets_IAT_cov
variance_fwd_packets_delta_len
variance_bwd_header_bytes_delta_len
max_bwd_header_bytes_delta_len
rst_flag_percentage_in_total
max_fwd_packets_delta_len
max_header_bytes_delta_len
bwd_ece_flag_percentage_in_tota